In [2]:
from pathlib import Path
import sys

proj_root = Path("/Users/jerry/coding/rrs-SDP-pigments")
sys.path.insert(0, str(proj_root))

In [3]:
import numpy as np
import pandas as pd

from utils.seabass_loader import load_hplc_data, extract_pigment_columns

In [4]:
pd.set_option("display.max_columns", None)

In [5]:
hplc_2024_11_30_path = proj_root / "experiments/pvst_sopace_validation/inputs/in_situ_hplc/940f89322d_PVST-SOPACE_KM2419-HPLC_20241130_R1.sb"
hplc_2025_10_14_path = proj_root / "experiments/pvst_sopace_validation/inputs/in_situ_hplc/10d10ff7e5_PVST_SOPACE_TN444-HPLC_20251014.sb"
hplc_2025_11_03_path = proj_root / "experiments/pvst_sopace_validation/inputs/in_situ_hplc/32c3f67fb5_PVST_SOPACE_TN440-HPLC_20251103.sb"

In [6]:
hplc_2024_11_30 = load_hplc_data(hplc_2024_11_30_path)
hplc_2025_10_14 = load_hplc_data(hplc_2025_10_14_path)
hplc_2025_11_03 = load_hplc_data(hplc_2025_11_03_path)

In [7]:
shapes = pd.Series({
    "hplc_2024_11_30": hplc_2024_11_30.shape,
    "hplc_2025_10_14": hplc_2025_10_14.shape,
    "hplc_2025_11_03": hplc_2025_11_03.shape,
})
shapes

hplc_2024_11_30    (49, 53)
hplc_2025_10_14    (68, 53)
hplc_2025_11_03    (13, 53)
dtype: object

In [8]:
def lon_lat_ranges(df):
    lon = pd.to_numeric(df['lon'])
    lat = pd.to_numeric(df['lat'])
    return pd.Series({
        'lon_min': lon.min(),
        'lon_max': lon.max(),
        'lat_min': lat.min(),
        'lat_max': lat.max(),
    })

ranges = pd.DataFrame({
    'hplc_2024_11_30': lon_lat_ranges(hplc_2024_11_30),
    'hplc_2025_10_14': lon_lat_ranges(hplc_2025_10_14),
    'hplc_2025_11_03': lon_lat_ranges(hplc_2025_11_03),
}).T
ranges


,lon_min,lon_max,lat_min,lat_max
hplc_2024_11_30,-157.69040,-132.3535,-14.2475,20.65700
hplc_2025_10_14,85.99957,88.5087,9.9134,13.94745
hplc_2025_11_03,148.20800,149.0300,7.6510,15.00000


In [9]:
def date_ranges(df):
    dates = pd.to_datetime(df['date'].astype('string'))
    return dates

date_ranges(hplc_2025_10_14)

0    2025-05-06
1    2025-05-06
2    2025-05-06
3    2025-05-09
4    2025-05-09
        ...    
63   2025-06-14
64   2025-06-14
65   2025-06-15
66   2025-06-15
67   2025-06-15
Name: date, Length: 68, dtype: datetime64[ns]

### 2025-10-14, Bay of Bengal

In [10]:
import rskit as rs
from rskit.plugins import NasaEarthdata
nasa = NasaEarthdata()
rs.plugins.get_params_schema('nasa_earthdata')

{'required_fields': ['collection_concept_id'],
 'optional_fields': ['cloud_cover',
  'sort_key',
  'max_granules',
  'variables',
  'drop_nan_lines'],
 'field_descriptions': {'collection_concept_id': 'CMR collection concept ID.',
  'cloud_cover': 'Tuple of (min_percent, max_percent) for cloud cover filtering.',
  'sort_key': "CMR sort key (e.g., '-start_date').",
  'max_granules': 'Maximum number of granules to return/download.',
  'variables': 'List of variable names to subset during client-side processing (Harmony ignores variable subsetting).',
  'drop_nan_lines': 'Drop all-NaN lines during client-side subsetting.'},
 'notes': ['Query.time(...) and Query.region(...) are required for NASA Earthdata downloads.']}

In [11]:
hplc_2024_11_30_bbox = (-157.6904, -14.2475, -132.3535, 20.657)
hplc_2025_10_14_bbox = (85.9995703, 9.9134, 88.5087, 13.94745)
hplc_2025_11_03_bbox = (148.208, 7.651, 149.03, 15.0)

collection_id = nasa.get_collection_concept_id(short_name='PACE_OCI_L2_AOP', version='3.1')

query = (
    rs.query(source='nasa_earthdata')
    .region(bbox=hplc_2025_10_14_bbox)
    .time(start='2025-05-06', end='2025-06-15')
    .with_params(collection_concept_id=collection_id)
)
nasa.download_subsetted_data(
    query,
    destination=Path('/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs'),
    mask_out_of_bounds=True
)

422 Client Error: Unprocessable Entity for url: https://harmony.earthdata.nasa.gov/C3620139598-OB_CLOUD/ogc-api-coverages/1.0.0/collections/all/coverage/rangeset?subset=lon%2885.9995703%3A88.5087%29&subset=lat%289.9134%3A13.94745%29&subset=time%28%222025-05-06T00%3A00%3A00Z%22%3A%222025-06-15T00%3A00%3A00Z%22%29


/Users/jerry/coding/RS-Kit/src/rskit/plugins/nasa_earthdata/base.py:297: UserWarning: Harmony subsetting failed, attemping to subset locally.
  warnings.warn("Harmony subsetting failed, attemping to subset locally.")


Skipping /Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250519T060350.L2.OC_AOP.V3_1.nc: there is either no spatial or no temporal overlap detected
Skipping /Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250608T075939.L2.OC_AOP.V3_1.nc: there is either no spatial or no temporal overlap detected


['/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250506T063633.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250507T071152.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250510T071924.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250511T061620.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250511T075441.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250512T065140.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_OCI.20250513T072658.L2.OC_AOP.V3_1.nc',
 '/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/rrs/PACE_

In [12]:
sst_id = 'C1615905770-OB_DAAC'
sss_id = 'C2208422957-POCLOUD'

In [11]:
hplc = extract_pigment_columns(hplc_2025_10_14)
hplc.notna().mean()

station      1.000000
date         1.000000
time         1.000000
lon          1.000000
lat          1.000000
Allo         0.000000
But-fuco     0.941176
Chl_c1c2     1.000000
Chl_c3       1.000000
DV_Chl_a     1.000000
Fuco         0.941176
Hex-fuco     1.000000
MV_Chl_b     1.000000
Neo          0.147059
Perid        1.000000
Tchl         1.000000
Tot_Chl_a    1.000000
Viola        0.588235
Zea          1.000000
dtype: float64

In [18]:
import xarray as xr
path = Path('/Users/jerry/coding/rrs-SDP-pigments/experiments/pvst_sopace_validation/inputs/sss/SMAP_L3_SSS_20250502_8DAYS_V5.0.nc')

with xr.open_dataset(path) as ds:
    print(ds)

<xarray.Dataset> Size: 37MB
Dimensions:               (latitude: 720, longitude: 1440, time: 1)
Coordinates:
  * latitude              (latitude) float32 3kB 89.88 89.62 ... -89.62 -89.88
  * longitude             (longitude) float32 6kB -179.9 -179.6 ... 179.6 179.9
  * time                  (time) datetime64[ns] 8B 2025-05-02T12:00:00
Data variables:
    smap_sss              (latitude, longitude) float32 4MB ...
    anc_sss               (latitude, longitude) float32 4MB ...
    anc_sst               (latitude, longitude) float32 4MB ...
    smap_spd              (latitude, longitude) float32 4MB ...
    smap_high_spd         (latitude, longitude) float32 4MB ...
    weight                (latitude, longitude) float32 4MB ...
    land_fraction         (latitude, longitude) float32 4MB ...
    ice_fraction          (latitude, longitude) float32 4MB ...
    smap_sss_uncertainty  (latitude, longitude) float32 4MB ...
Attributes: (12/40)
    title:                       SMAP 0.25x0.25 d

Fix wavelength extraction for real OCI L2 files (current hard blocker)
Right now the pipeline fails before producing any matchups because _get_l2_rrs_wavelengths bails out too early when it sees sensor_band_parameters/wavelength (286 bands) instead of sensor_band_parameters/wavelength_3d (172 bands for Rrs). The relevant code is pace_l2.py (line 94).
That means extract_l2_matchup(...) cannot interpolate to the 1 nm grid, so PVST can’t run end-to-end yet.
Make SST/SSS sampling runnable in swot-pace (and time-aware)
main.py currently uses xr.open_mfdataset(...) for SST/SSS (main.py (line 218)). In your swot-pace env, xarray.open_mfdataset is erroring because dask isn’t available (and it also tends to assume a “time” coordinate exists for multi-file combining).
Your current PVST downloads look like 2D lat/lon slices with no time dim (e.g., SSS and SST files are lat/lon only), so _sample_gridded_field(..., time=...) in main.py (line 52) can’t do a meaningful temporal “nearest” selection anyway.
You need a decision here:
either: parse each ancillary filename’s date/window and select the best file per matchup time (recommended if the files are 2D-only),
or: change the download method so the ancillary files retain a time coordinate that xarray can select.
Confirm SST product is actually what you want (daily vs 8‑day)
Your stated requirement says “Aqua MODIS Level 3 SST (Daily Mapped), 4 km”.
The SST files currently in PVST inputs look like L3m.8D... composites, not daily. If daily is required for your physics (or just to match the paper’s intent), you still need to adjust the Earthaccess search/filtering in tmp_download_pvst_sst_sss.py (line 160) so it only pulls daily granules.
Tighten the “Kramer preprocessing” to match your written spec (possible methodology gap)
Your description says smoothing is applied to the residual before taking the second derivative.
In the current code path, smoothing happens at L2 spectrum extraction time (pace_l2.py (line 151)), but run_sdp takes the second derivative of the residuals without an explicit residual-smoothing step (prediction.py (line 102)).
You should decide where smoothing belongs (on Rrs_obs, on δRrs, or both), then implement it consistently.
Also: run_sdp still prints the full derivative matrix (prediction.py (line 105)), which will spam logs and slow runs; that should be removed once you’re done debugging.
Add the “evaluation” outputs (plots + metrics) for matchups
main.py writes predictions + matchup metadata, but it does not embed the in-situ HPLC pigment values into sdp_results.nc (main.py (line 260) only writes predictions + rrs + obs/pixel coords).
You still need:
a mapping from HPLC pigment column names → model pigment names,
parity plots + Bland–Altman plots,
RMSE / MAD / bias computation (per pigment),
a script/notebook to generate those artifacts from the matchup-style sdp_results.nc.
You can reuse plotting/stat patterns from pigment_comparison 2.py (line 211), but it currently assumes gridded lat/lon outputs; you’ll need a matchup-based version.
Run PVST end-to-end and sanity-check
After (1)–(3), you should be able to actually produce sdp_results.nc (currently that folder is empty).
Then validate:
how many matchups survived QC/time/distance,
distributions of distance_km, sst, sss,
that wavelengths are aligned to 400–700 nm @ 1 nm before run_sdp.